# PropIQ — Week 7: Gold Model, KPIs and Reconciliation

**Project:** P15 PropIQ — Real Estate Market Analytics
**Notebook path:** `notebooks/05_gold_aggregations.ipynb`
**Authority:** `P15_PropIQ_Student_Project_Playbook_v1_0_APPROVED.pdf` — Gold Model
and KPI Contract (playbook page 20), Week 07 section (playbook pages 54–59)
**Method reference:** ZENAIZ PageLoop Week-7 *Gold Table Design and Build*
notebook — used only for the learning flow (Question → KPI contract → Grain
+ scope → Trusted inputs → join safety → aggregate → validate → reconcile →
rerun). No PageLoop table names, KPIs or grains are carried over.

> **Week-7 job (per the approved playbook):** build separate `fact_listing`
> and `fact_lead` tables, the seven governed dimensions, the five approved
> summaries and the eight approved KPI definitions — from **Trusted Silver
> only**. Protect listing grain from lead fan-out. Handle zero denominators
> explicitly. Spot-check at least one locality, broker and listing manually.

**Inputs used (confirmed from your actual Week-6 notebook, not assumed):**
`silver_listings_trusted`, `silver_leads_trusted`, `silver_localities_trusted`,
`silver_brokers_trusted` — each carries every Candidate business column
forward (`SELECT l.*, ...`) plus the DQ check columns, `failed_rule_ids`,
`dq_status`, `highest_severity`, `dq_checked_at`, `dq_ruleset_version`.
Quarantine tables are never read in this notebook — Gold reads Trusted only.


## 1. Outcome first — what this notebook must produce

Per the playbook's Gold Model and KPI Contract (page 20):

| Object group | Objects this notebook builds |
|---|---|
| Dimensions (7) | `dim_date`, `dim_locality`, `dim_property`, `dim_broker`, `dim_listing_status`, `dim_price_band`, `dim_lead_channel` |
| Facts (2 of 4) | `fact_listing`, `fact_lead` — `fact_listing_status_event` and `fact_listing_stream` are Week-10 streaming facts and are explicitly out of scope here |
| Summaries (5) | `locality_price_summary`, `listing_performance_summary`, `lead_conversion_summary`, `broker_performance_summary`, `inventory_age_summary` |
| KPIs (8) | Active Listings, Median Listing Price, Median Price per Sq Ft, Lead Conversion Rate, Average Leads per Listing, Average Days on Market, Stale Listing Rate, DQ Pass Rate |

**Grains declared up front** (so nothing is guessed mid-build):

| Object | One row represents |
|---|---|
| `fact_listing` | one Trusted physical listing (`record_uid`) |
| `fact_lead` | one Trusted physical lead (`record_uid`) |
| `dim_locality` | one Trusted locality (`locality_id`) |
| `dim_broker` | one Trusted broker (`broker_id`) |
| `dim_property` | one distinct `(property_type, furnishing)` combination observed in Trusted listings |
| `dim_listing_status` | one distinct `listing_status` value observed in Trusted listings |
| `dim_price_band` | one documented price band (see Section 7) |
| `dim_lead_channel` | one distinct `lead_channel` value observed in Trusted leads |
| `dim_date` | one calendar date spanning the dates actually used across Trusted listings/leads |
| `locality_price_summary`, `broker_performance_summary` | one row per `locality_id` / `broker_id` |
| `listing_performance_summary`, `inventory_age_summary`, `lead_conversion_summary` | one row per `locality_id` (a documented design choice — see Section 12; confirm with mentors if a different grain was approved) |

**A note on two undocumented parameters:** the playbook's KPI Catalogue
requires a documented **stale-listing age threshold** and a documented
**zero-denominator policy**, but does not publish either value. Rather than
inventing them silently, Section 11 declares them as **editable parameters**
with the assumption stated in plain text — confirm both with mentors and
record the final decision in `docs/gold_metrics_definition.md` before this
notebook's KPI outputs are treated as final.


## 2. Week 7 in one minute

| Concept | Meaning |
|---|---|
| Trusted Silver | the only permitted Gold input — never read Quarantine |
| Dimension | governed, deduplicated descriptive context (who/what/when) |
| Fact | one row per physical business event/entity, at a declared grain |
| Summary | a pre-aggregated, business-ready rollup built from facts + dimensions |
| KPI contract | business question, formula, grain, eligible rows, exclusions, zero-denominator policy — written **before** any aggregation SQL |

**Flow used throughout:** Question → KPI/object contract → Grain + scope →
Trusted inputs → join-safety proof → aggregate → validate → reconcile →
rerun proof.


## 3. Confirm the Week-6 handoff

Week 7 reads Trusted Silver only. It must not recreate Week 5 transformations
or Week 6 routing.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema()  AS active_schema;


active_catalog,active_schema
workspace,default


**Expected result:** one row showing the intended catalog/schema. Stop and correct if wrong.

In [0]:
%sql
SHOW TABLES LIKE '*trusted*';


database,tableName,isTemporary
default,silver_brokers_trusted,false
default,silver_leads_trusted,false
default,silver_listings_trusted,false
default,silver_localities_trusted,false


**Expected result:** `silver_listings_trusted`, `silver_leads_trusted`,
`silver_localities_trusted`, `silver_brokers_trusted`. Stop and complete
Week 6 first if any is missing.

### 3.1 Reconfirm Week-6 reconciliation before building on top of it

In [0]:
%sql
WITH reconciliation AS (
  SELECT 'listings' AS entity,
         (SELECT COUNT(DISTINCT record_uid) FROM silver_propiq_listings_candidate) AS candidate_rows,
         (SELECT COUNT(DISTINCT record_uid) FROM silver_listings_trusted) AS trusted_rows,
         (SELECT COUNT(DISTINCT record_uid) FROM quarantine_listings) AS quarantine_rows
  UNION ALL
  SELECT 'leads',
         (SELECT COUNT(DISTINCT record_uid) FROM silver_propiq_leads_candidate),
         (SELECT COUNT(DISTINCT record_uid) FROM silver_leads_trusted),
         (SELECT COUNT(DISTINCT record_uid) FROM quarantine_leads)
  UNION ALL
  SELECT 'localities',
         (SELECT COUNT(DISTINCT record_uid) FROM silver_propiq_localities_candidate),
         (SELECT COUNT(DISTINCT record_uid) FROM silver_localities_trusted),
         (SELECT COUNT(DISTINCT record_uid) FROM quarantine_localities)
  UNION ALL
  SELECT 'brokers',
         (SELECT COUNT(DISTINCT record_uid) FROM silver_propiq_brokers_candidate),
         (SELECT COUNT(DISTINCT record_uid) FROM silver_brokers_trusted),
         (SELECT COUNT(DISTINCT record_uid) FROM quarantine_brokers)
)
SELECT *,
       CASE WHEN candidate_rows = trusted_rows + quarantine_rows
            THEN 'PASS' ELSE 'CHECK' END AS reconciliation_status
FROM reconciliation;


entity,candidate_rows,trusted_rows,quarantine_rows,reconciliation_status
listings,50200,49000,1200,PASS
leads,120800,118000,2800,PASS
localities,80,80,0,PASS
brokers,320,320,0,PASS


**Expected result:** `PASS` for all four entities. Resolve any `CHECK` in Week 6 before continuing — Gold does not repair Week-6 routing.

### 3.2 Inspect only the fields this notebook needs

Gold should not copy every Silver column. Preview the minimum fields first.

In [0]:
%sql
SELECT record_uid, listing_id, locality_id, broker_id, property_type, furnishing,
       listing_status, asking_price_inr, built_up_area_sqft, price_per_sqft,
       listing_created_date, completion_date, is_completed, is_chronology_valid
FROM silver_listings_trusted
LIMIT 10;


record_uid,listing_id,locality_id,broker_id,property_type,furnishing,listing_status,asking_price_inr,built_up_area_sqft,price_per_sqft,listing_created_date,completion_date,is_completed,is_chronology_valid
LISTING-PHY-0000001,LST-0000001,LOC-062,BRK-0191,Apartment,Furnished,active,11005988,1631,6748,2025-05-15,null,false,null
LISTING-PHY-0000002,LST-0000002,LOC-056,BRK-0130,Apartment,Semi-furnished,active,21793252,1157,18836,2025-07-17,null,false,null
LISTING-PHY-0000003,LST-0000003,LOC-045,BRK-0215,Apartment,Semi-furnished,rented,16873096,1973,8552,2024-12-23,2025-03-01,true,true
LISTING-PHY-0000004,LST-0000004,LOC-036,BRK-0177,Apartment,Furnished,paused,21144060,1164,18165,2024-10-31,null,false,null
LISTING-PHY-0000005,LST-0000005,LOC-041,BRK-0298,Apartment,Furnished,sold,9571528,1478,6476,2025-07-28,2025-12-04,true,true
LISTING-PHY-0000006,LST-0000006,LOC-014,BRK-0050,Apartment,Unfurnished,sold,10529001,1827,5763,2024-05-25,2024-09-26,true,true
LISTING-PHY-0000007,LST-0000007,LOC-029,BRK-0055,Villa,Semi-furnished,expired,34199100,4740,7215,2025-10-02,null,false,null
LISTING-PHY-0000008,LST-0000008,LOC-074,BRK-0011,Apartment,Semi-furnished,active,12205728,1296,9418,2024-01-30,null,false,null
LISTING-PHY-0000009,LST-0000009,LOC-046,BRK-0017,Apartment,Unfurnished,rented,9395822,1301,7222,2025-02-04,2025-06-09,true,true
LISTING-PHY-0000010,LST-0000010,LOC-071,BRK-0201,Villa,Semi-furnished,sold,39615810,3882,10205,2024-02-12,2024-05-01,true,true


In [0]:
%sql
SELECT record_uid, lead_id, listing_id, lead_channel, buyer_intent,
       qualified_flag, lead_status, lead_timestamp
FROM silver_leads_trusted
LIMIT 10;


record_uid,lead_id,listing_id,lead_channel,buyer_intent,qualified_flag,lead_status,lead_timestamp
LEAD-PHY-0000001,LED-00000001,LST-0018000,Campaign,Medium,false,new,2026-05-25T18:19:50.000Z
LEAD-PHY-0000002,LED-00000002,LST-0024708,Portal Search,Medium,true,qualified,2025-10-22T21:10:21.000Z
LEAD-PHY-0000003,LED-00000003,LST-0020315,Portal Search,High,false,new,2026-04-17T01:24:48.000Z
LEAD-PHY-0000004,LED-00000004,LST-0015485,Partner,Medium,true,negotiation,2025-03-01T21:32:16.000Z
LEAD-PHY-0000005,LED-00000005,LST-0037002,Portal Search,Medium,true,qualified,2025-06-18T04:22:06.000Z
LEAD-PHY-0000006,LED-00000006,LST-0019134,Portal Search,Exploratory,true,negotiation,2026-02-14T18:22:41.000Z
LEAD-PHY-0000007,LED-00000007,LST-0009732,Portal Search,Medium,true,site_visit,2026-05-17T07:00:28.000Z
LEAD-PHY-0000008,LED-00000008,LST-0004899,Portal Search,Medium,false,closed_unqualified,2026-04-18T01:45:58.000Z
LEAD-PHY-0000009,LED-00000009,LST-0005907,Walk-in,Exploratory,true,negotiation,2026-02-24T22:13:22.000Z
LEAD-PHY-0000010,LED-00000010,LST-0017045,Portal Search,Exploratory,true,qualified,2025-02-27T15:19:28.000Z


## 4. Prove every lookup key is safe before joining

A lookup with duplicate keys would multiply `fact_listing` or `fact_lead`
rows during a join. Week 6 already enforced these uniqueness rules
(`MK-LOC-02`, `MK-BRK-02`, `P15-DQ-01`), but Week 7 reconfirms the join
assumption independently rather than trusting it silently — the playbook's
own realistic-failure scenario for this week is exactly a broker appearing
to have thousands of listings because a fact was joined to lead rows without
this kind of check.

In [0]:
%sql
SELECT locality_id, COUNT(*) AS occurrences
FROM silver_localities_trusted
GROUP BY locality_id
HAVING COUNT(*) > 1;


locality_id,occurrences


In [0]:
%sql
SELECT broker_id, COUNT(*) AS occurrences
FROM silver_brokers_trusted
GROUP BY broker_id
HAVING COUNT(*) > 1;


broker_id,occurrences


In [0]:
%sql
SELECT listing_id, COUNT(*) AS occurrences
FROM silver_listings_trusted
WHERE listing_id IS NOT NULL
GROUP BY listing_id
HAVING COUNT(*) > 1;


listing_id,occurrences


**Expected result:** zero rows from all three checks. If any returns rows, stop — repair Week 6 routing before building Gold, do not silently dedupe here.

## 5. Build the seven governed dimensions

Each dimension is built directly from Trusted Silver, deduplicated by its
declared natural key. No dimension invents a value that is not already
present in Trusted data.

### 5.1 `dim_locality` (grain: one row per Trusted `locality_id`)

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_locality
USING DELTA
AS
SELECT
  locality_id,
  locality_name,
  city,
  city_zone,
  market_segment,
  reference_price_per_sqft,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM silver_localities_trusted;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS dim_locality_rows,
       COUNT(DISTINCT locality_id) AS distinct_locality_id
FROM gold_dim_locality;


dim_locality_rows,distinct_locality_id
80,80


**Expected result:** the two counts are equal — one row per `locality_id`, no duplicates introduced.

### 5.2 `dim_broker` (grain: one row per Trusted `broker_id`)

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_broker
USING DELTA
AS
SELECT
  broker_id,
  agency_name,
  city,
  broker_tier,
  service_rating,
  onboarded_date,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM silver_brokers_trusted;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS dim_broker_rows,
       COUNT(DISTINCT broker_id) AS distinct_broker_id
FROM gold_dim_broker;


dim_broker_rows,distinct_broker_id
320,320


### 5.3 `dim_property` (grain: one row per distinct `(property_type, furnishing)` in Trusted Listings)

This is a design choice, not a published playbook grain — the playbook names
`dim_property` as an object but does not define its exact attributes. Using
the two descriptive attributes that recur across listings (rather than
per-listing details like `bedrooms`/`built_up_area_sqft`, which stay in
`fact_listing`) keeps this a small, genuinely reusable dimension. Confirm
with mentors if a different definition was approved.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_property
USING DELTA
AS
SELECT
  row_number() OVER (ORDER BY property_type, furnishing) AS property_key,
  property_type,
  furnishing,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM (
  SELECT DISTINCT property_type, furnishing
  FROM silver_listings_trusted
);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_dim_property ORDER BY property_key;


property_key,property_type,furnishing,_gold_created_at,_gold_schema_version
1,Apartment,Furnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
2,Apartment,Semi-furnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
3,Apartment,Unfurnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
4,Row House,Furnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
5,Row House,Semi-furnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
6,Row House,Unfurnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
7,Studio,Furnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
8,Studio,Semi-furnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
9,Studio,Unfurnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0
10,Unknown Tower,Furnished,2026-09-05T14:02:53.022Z,propiq_gold_v1.0


### 5.4 `dim_listing_status` (grain: one row per distinct Trusted `listing_status`)

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_listing_status
USING DELTA
AS
SELECT
  row_number() OVER (ORDER BY listing_status) AS listing_status_key,
  listing_status,
  listing_status IN ('sold', 'rented') AS is_completed_status,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM (SELECT DISTINCT listing_status FROM silver_listings_trusted);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_dim_listing_status ORDER BY listing_status_key;


listing_status_key,listing_status,is_completed_status,_gold_created_at,_gold_schema_version
1,active,false,2026-09-05T14:02:58.398Z,propiq_gold_v1.0
2,expired,false,2026-09-05T14:02:58.398Z,propiq_gold_v1.0
3,paused,false,2026-09-05T14:02:58.398Z,propiq_gold_v1.0
4,rented,true,2026-09-05T14:02:58.398Z,propiq_gold_v1.0
5,sold,true,2026-09-05T14:02:58.398Z,propiq_gold_v1.0
6,withdrawn,false,2026-09-05T14:02:58.398Z,propiq_gold_v1.0


### 5.5 `dim_price_band` (grain: one row per documented price band)

**Declared assumption — not published in the playbook.** The KPI Catalogue
requires price-band context but does not publish the band boundaries. The
bands below are seeded from the P15-DQ-03 approved price range (₹2,000,000–
₹150,000,000) split into round, explainable bands. Confirm the final bands
with mentors and update this table (and re-run `fact_listing` in Section 6)
if they differ.

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_price_band
USING DELTA
AS
SELECT * FROM (VALUES
  (1, 'Under 5M',   0,          4999999),
  (2, '5M-10M',     5000000,    9999999),
  (3, '10M-20M',    10000000,   19999999),
  (4, '20M-50M',    20000000,   49999999),
  (5, '50M-150M',   50000000,   150000000)
) AS bands(price_band_key, price_band, band_min_inr, band_max_inr);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_dim_price_band ORDER BY price_band_key;


price_band_key,price_band,band_min_inr,band_max_inr
1,Under 5M,0,4999999
2,5M-10M,5000000,9999999
3,10M-20M,10000000,19999999
4,20M-50M,20000000,49999999
5,50M-150M,50000000,150000000


### 5.6 `dim_lead_channel` (grain: one row per distinct Trusted `lead_channel`)

In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_lead_channel
USING DELTA
AS
SELECT
  row_number() OVER (ORDER BY lead_channel) AS lead_channel_key,
  lead_channel,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM (SELECT DISTINCT lead_channel FROM silver_leads_trusted);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_dim_lead_channel ORDER BY lead_channel_key;


lead_channel_key,lead_channel,_gold_created_at,_gold_schema_version
1,Broker Referral,2026-09-05T14:03:08.184Z,propiq_gold_v1.0
2,Campaign,2026-09-05T14:03:08.184Z,propiq_gold_v1.0
3,Partner,2026-09-05T14:03:08.184Z,propiq_gold_v1.0
4,Portal Search,2026-09-05T14:03:08.184Z,propiq_gold_v1.0
5,Unverified Scrape,2026-09-05T14:03:08.184Z,propiq_gold_v1.0
6,Walk-in,2026-09-05T14:03:08.184Z,propiq_gold_v1.0


### 5.7 `dim_date` (grain: one row per calendar date spanning Trusted listings/leads activity)

Built as a date spine covering every date actually referenced by Trusted
listings (`listing_created_date`, `completion_date`) and Trusted leads
(`lead_timestamp`), plus today — so `fact_listing`/`fact_lead` never need a
date outside this dimension's range.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW date_bounds AS
SELECT
  least(
    (SELECT MIN(listing_created_date) FROM silver_listings_trusted),
    (SELECT MIN(CAST(completion_date AS DATE)) FROM silver_listings_trusted WHERE completion_date IS NOT NULL),
    (SELECT MIN(CAST(lead_timestamp AS DATE)) FROM silver_leads_trusted WHERE lead_timestamp IS NOT NULL)
  ) AS min_date,
  greatest(
    (SELECT MAX(listing_created_date) FROM silver_listings_trusted),
    (SELECT MAX(CAST(completion_date AS DATE)) FROM silver_listings_trusted WHERE completion_date IS NOT NULL),
    (SELECT MAX(CAST(lead_timestamp AS DATE)) FROM silver_leads_trusted WHERE lead_timestamp IS NOT NULL),
    current_date()
  ) AS max_date;


In [0]:
%sql
CREATE OR REPLACE TABLE gold_dim_date
USING DELTA
AS
SELECT
  calendar_date AS date_key,
  year(calendar_date) AS year,
  quarter(calendar_date) AS quarter,
  month(calendar_date) AS month,
  date_format(calendar_date, 'MMMM') AS month_name,
  day(calendar_date) AS day_of_month,
  date_format(calendar_date, 'EEEE') AS day_name,
  dayofweek(calendar_date) IN (1, 7) AS is_weekend,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM (
  SELECT explode(sequence(min_date, max_date, interval 1 day)) AS calendar_date
  FROM date_bounds
);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS dim_date_rows, MIN(date_key) AS earliest, MAX(date_key) AS latest
FROM gold_dim_date;


dim_date_rows,earliest,latest
979,2024-01-01,2026-09-05


**Expected result:** a real row count and date range covering every date used by Trusted listings/leads through today.

## 6. Build `fact_listing` (grain: one row per Trusted physical listing, `record_uid`)

This is the highest-risk build in Week 7: the playbook's realistic failure
for this week is exactly a broker appearing to have thousands of listings
because facts were joined to lead rows. `fact_listing` is built **without
ever joining to `silver_leads_trusted`** — lead information belongs only in
`fact_lead`.

### 6.1 Contract

| Item | Value |
|---|---|
| Grain | one row per Trusted listing (`record_uid`) |
| Input | `silver_listings_trusted` only |
| Lookups | `dim_property` (property_type, furnishing), `dim_price_band` (asking_price_inr range) |
| New measure | `days_on_market_status_aware` — status-aware end date: `actual_days_on_market` when `is_completed`, otherwise days from `listing_created_date` to today |


### 6.2 Build the enriched, row-preserving view

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fact_listing_input AS
SELECT
  l.record_uid,
  l.listing_id,
  l.locality_id,
  l.broker_id,
  l.listing_status,
  l.asking_price_inr,
  l.built_up_area_sqft,
  l.price_per_sqft,
  l.calculated_price_per_sqft,
  l.price_per_sqft_variance,
  l.listing_created_date,
  l.completion_date,
  l.last_updated_timestamp,
  l.is_completed,
  l.is_chronology_valid,
  l.actual_days_on_market,
  l.days_since_last_update,
  p.property_key,
  b.price_band_key
FROM silver_listings_trusted l
LEFT JOIN gold_dim_property p
  ON l.property_type = p.property_type AND l.furnishing = p.furnishing
LEFT JOIN gold_dim_price_band b
  ON l.asking_price_inr BETWEEN b.band_min_inr AND b.band_max_inr;


### 6.3 Validate the join before using it

In [0]:
%sql
WITH join_checks AS (
  SELECT
    (SELECT COUNT(*) FROM silver_listings_trusted) AS before_join_rows,
    (SELECT COUNT(*) FROM fact_listing_input) AS after_join_rows,
    (SELECT COUNT(*) FROM fact_listing_input WHERE property_key IS NULL) AS unmatched_property_rows,
    (SELECT COUNT(*) FROM fact_listing_input WHERE price_band_key IS NULL) AS unmatched_price_band_rows
)
SELECT *,
       CASE WHEN before_join_rows = after_join_rows
            THEN 'PASS' ELSE 'CHECK' END AS join_status
FROM join_checks;


before_join_rows,after_join_rows,unmatched_property_rows,unmatched_price_band_rows,join_status
49000,49000,0,0,PASS


**Expected result:** `before_join_rows = after_join_rows` (`PASS`). `unmatched_price_band_rows` should be zero given the P15-DQ-03 range enforcement; a nonzero `unmatched_property_rows` means a listing has a null `property_type`/`furnishing` that DQ-07 should have caught — investigate rather than silently proceeding.

In [0]:
%sql
SELECT record_uid, COUNT(*) AS occurrences
FROM fact_listing_input
GROUP BY record_uid
HAVING COUNT(*) > 1;


record_uid,occurrences


**Expected result:** zero rows. If not, stop — a join above is multiplying listings.

### 6.4 Add the status-aware days-on-market measure

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fact_listing_ready AS
SELECT *,
  CASE WHEN is_completed = true THEN actual_days_on_market
       ELSE datediff(current_date(), listing_created_date)
  END AS days_on_market_status_aware
FROM fact_listing_input;


### 6.5 Create the Gold Delta table

In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_listing
USING DELTA
AS
SELECT
  record_uid, listing_id, locality_id, broker_id, property_key, price_band_key,
  listing_status, asking_price_inr, built_up_area_sqft, price_per_sqft,
  calculated_price_per_sqft, price_per_sqft_variance,
  listing_created_date, completion_date, last_updated_timestamp,
  is_completed, is_chronology_valid,
  actual_days_on_market, days_since_last_update, days_on_market_status_aware,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM fact_listing_ready;


num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE TABLE gold_fact_listing;


col_name,data_type,comment
record_uid,string,null
listing_id,string,null
locality_id,string,null
broker_id,string,null
property_key,int,null
price_band_key,int,null
listing_status,string,null
asking_price_inr,bigint,null
built_up_area_sqft,int,null
price_per_sqft,bigint,null


In [0]:
%sql
SELECT COUNT(*) AS fact_listing_rows,
       COUNT(DISTINCT record_uid) AS distinct_record_uid
FROM gold_fact_listing;


fact_listing_rows,distinct_record_uid
49000,49000


**Expected result:** the two counts are equal, and both equal the Trusted Listings count from Section 3.1 — `fact_listing` preserves listing grain exactly.

### 6.6 Reference control — every populated dimension key resolves

In [0]:
%sql
SELECT f.* FROM gold_fact_listing f
LEFT ANTI JOIN gold_dim_locality d ON f.locality_id = d.locality_id
WHERE f.locality_id IS NOT NULL;


record_uid,listing_id,locality_id,broker_id,property_key,price_band_key,listing_status,asking_price_inr,built_up_area_sqft,price_per_sqft,calculated_price_per_sqft,price_per_sqft_variance,listing_created_date,completion_date,last_updated_timestamp,is_completed,is_chronology_valid,actual_days_on_market,days_since_last_update,days_on_market_status_aware,_gold_created_at,_gold_schema_version


In [0]:
%sql
SELECT f.* FROM gold_fact_listing f
LEFT ANTI JOIN gold_dim_broker d ON f.broker_id = d.broker_id
WHERE f.broker_id IS NOT NULL;


record_uid,listing_id,locality_id,broker_id,property_key,price_band_key,listing_status,asking_price_inr,built_up_area_sqft,price_per_sqft,calculated_price_per_sqft,price_per_sqft_variance,listing_created_date,completion_date,last_updated_timestamp,is_completed,is_chronology_valid,actual_days_on_market,days_since_last_update,days_on_market_status_aware,_gold_created_at,_gold_schema_version


**Expected result:** zero rows from both anti-joins — every populated `locality_id`/`broker_id` in `fact_listing` resolves to a governed dimension row.

## 7. Build `fact_lead` (grain: one row per Trusted physical lead, `record_uid`)

### 7.1 Contract

| Item | Value |
|---|---|
| Grain | one row per Trusted lead (`record_uid`) |
| Input | `silver_leads_trusted` only |
| Lookups | `dim_lead_channel`, `fact_listing` (for `listing_id` — proven unique in Section 4) |
| Reference control | every `fact_lead.listing_id` must resolve to `fact_listing.listing_id`, since Week 6's P15-DQ-06 already enforced this against Trusted Listings |


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fact_lead_input AS
SELECT
  ld.record_uid,
  ld.lead_id,
  ld.listing_id,
  ld.lead_channel,
  ld.buyer_intent,
  ld.qualified_flag,
  ld.lead_status,
  ld.budget_band,
  ld.lead_timestamp,
  lc.lead_channel_key,
  fl.locality_id AS listing_locality_id,
  fl.broker_id AS listing_broker_id
FROM silver_leads_trusted ld
LEFT JOIN gold_dim_lead_channel lc ON ld.lead_channel = lc.lead_channel
LEFT JOIN gold_fact_listing fl ON ld.listing_id = fl.listing_id;


### 7.2 Validate the join before using it

In [0]:
%sql
WITH join_checks AS (
  SELECT
    (SELECT COUNT(*) FROM silver_leads_trusted) AS before_join_rows,
    (SELECT COUNT(*) FROM fact_lead_input) AS after_join_rows,
    (SELECT COUNT(*) FROM fact_lead_input WHERE listing_locality_id IS NULL) AS unmatched_listing_rows
)
SELECT *,
       CASE WHEN before_join_rows = after_join_rows
            THEN 'PASS' ELSE 'CHECK' END AS join_status
FROM join_checks;


before_join_rows,after_join_rows,unmatched_listing_rows,join_status
118000,118000,0,PASS


**Expected result:** `before_join_rows = after_join_rows` and `unmatched_listing_rows = 0` — P15-DQ-06 already guaranteed every Trusted lead's `listing_id` resolves to a Trusted listing, so this reconfirms rather than repairs.

In [0]:
%sql
SELECT record_uid, COUNT(*) AS occurrences
FROM fact_lead_input
GROUP BY record_uid
HAVING COUNT(*) > 1;


record_uid,occurrences


**Expected result:** zero rows.

### 7.3 Create the Gold Delta table

In [0]:
%sql
CREATE OR REPLACE TABLE gold_fact_lead
USING DELTA
AS
SELECT
  record_uid, lead_id, listing_id, lead_channel_key,
  buyer_intent, qualified_flag, lead_status, budget_band, lead_timestamp,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM fact_lead_input;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS fact_lead_rows, COUNT(DISTINCT record_uid) AS distinct_record_uid
FROM gold_fact_lead;


fact_lead_rows,distinct_record_uid
118000,118000


**Expected result:** the two counts are equal, and both equal the Trusted Leads count from Section 3.1.

## 8. The eight approved KPI contracts (from the playbook's KPI Catalogue)

Every KPI below is implemented **after** its contract is stated — numerator,
denominator/scope, exclusions and zero-denominator handling — per the
playbook's own Week-7 objective: *"Implement KPI numerators, denominators
and exclusions... Handle zero denominators explicitly."*

| # | KPI | Formula meaning (from the playbook) | Control (from the playbook) |
|---|---|---|---|
| 1 | Active Listings | distinct trusted listings with active status | exclude quarantine and unresolved duplicates (already true — source is `fact_listing`, built only from Trusted) |
| 2 | Median Listing Price | median normalized asking price | calculate at listing grain |
| 3 | Median Price per Sq Ft | median trusted `price_per_sqft` | do not average raw ratios after fan-out — computed directly from `fact_listing`, never from a joined/grouped table |
| 4 | Lead Conversion Rate | closed listings after qualified lead / listings with qualified leads × 100 | return 0 or blank by documented zero-denominator policy |
| 5 | Average Leads per Listing | trusted leads / listings receiving leads | denominator excludes zero-lead listings |
| 6 | Average Days on Market | completion or current-age days from creation | use status-aware end date (`days_on_market_status_aware`) |
| 7 | Stale Listing Rate | active listings above approved age threshold / active listings × 100 | threshold documented and filter-aware |
| 8 | DQ Pass Rate | Trusted evaluated rows / total evaluated input rows × 100 | report by source file and batch |


### 8.1 Declared parameters (confirm with mentors, then keep these values in sync with `docs/gold_metrics_definition.md`)

- **Zero-denominator policy:** this notebook returns `NULL` (blank) when a
  KPI's denominator is zero, using `NULLIF(denominator, 0)`. This is a
  documented choice, not the only valid one — if mentors approve returning
  `0` instead, replace `NULLIF(denominator, 0)` with
  `COALESCE(numerator / NULLIF(denominator, 0), 0)` throughout Section 9.
- **Stale-listing age threshold:** not published by the playbook. This
  notebook uses **90 days** as a placeholder, set as an editable SQL
  variable below — confirm the real approved threshold before treating
  Stale Listing Rate as final.

In [0]:
%sql
DECLARE OR REPLACE VARIABLE stale_listing_threshold_days INT = 90;


If your Databricks runtime does not support `DECLARE VARIABLE`, replace `stale_listing_threshold_days` in Section 9.7 with the literal `90` (or your mentor-approved value) directly.

## 9. Compute the eight KPIs

### 9.1 KPI 1 — Active Listings

**Numerator:** distinct Trusted listings with `listing_status = 'active'`.
No separate denominator — this is a count, not a rate. Quarantine and
unresolved duplicates are already excluded because the source is
`fact_listing`, built only from Trusted Silver.

In [0]:
%sql
SELECT COUNT(DISTINCT record_uid) AS active_listings
FROM gold_fact_listing
WHERE listing_status = 'active';


active_listings
23406


**Run and record actual result.**

### 9.2 KPI 2 — Median Listing Price

**Grain:** one value per listing (`asking_price_inr`), before any join or
groupby that could change row count.

In [0]:
%sql
SELECT percentile_approx(asking_price_inr, 0.5) AS median_listing_price
FROM gold_fact_listing;


median_listing_price
14341684


### 9.3 KPI 3 — Median Price per Sq Ft

Computed directly from `fact_listing.price_per_sqft` — never averaged after
a join to `fact_lead` or any other table that would fan out listing rows.

In [0]:
%sql
SELECT percentile_approx(price_per_sqft, 0.5) AS median_price_per_sqft
FROM gold_fact_listing;


median_price_per_sqft
9416


### 9.4 KPI 4 — Lead Conversion Rate

**Numerator:** distinct listings that are both completed (`is_completed`)
and received at least one qualified lead.
**Denominator:** distinct listings that received at least one qualified
lead (not all listings — per the formula's own wording, "listings with
qualified leads").
**Zero-denominator policy:** `NULL` when no listing received a qualified lead.

In [0]:
%sql
WITH qualified_lead_listings AS (
  SELECT DISTINCT listing_id
  FROM gold_fact_lead
  WHERE qualified_flag = true
),
numerator AS (
  SELECT COUNT(DISTINCT fl.record_uid) AS closed_after_qualified_lead
  FROM gold_fact_listing fl
  JOIN qualified_lead_listings q ON fl.listing_id = q.listing_id
  WHERE fl.is_completed = true
),
denominator AS (
  SELECT COUNT(DISTINCT listing_id) AS listings_with_qualified_leads
  FROM qualified_lead_listings
)
SELECT
  n.closed_after_qualified_lead,
  d.listings_with_qualified_leads,
  ROUND(100.0 * n.closed_after_qualified_lead / NULLIF(d.listings_with_qualified_leads, 0), 2)
    AS lead_conversion_rate_pct
FROM numerator n CROSS JOIN denominator d;


closed_after_qualified_lead,listings_with_qualified_leads,lead_conversion_rate_pct
8010,28379,28.23


### 9.5 KPI 5 — Average Leads per Listing

**Numerator:** count of Trusted leads.
**Denominator:** distinct listings that received at least one lead
(zero-lead listings are excluded from the denominator, per the formula's
own wording).

In [0]:
%sql
WITH listings_with_leads AS (
  SELECT DISTINCT listing_id FROM gold_fact_lead
)
SELECT
  (SELECT COUNT(*) FROM gold_fact_lead) AS trusted_lead_count,
  (SELECT COUNT(*) FROM listings_with_leads) AS listings_receiving_leads,
  ROUND(
    (SELECT COUNT(*) FROM gold_fact_lead) * 1.0
    / NULLIF((SELECT COUNT(*) FROM listings_with_leads), 0), 2
  ) AS average_leads_per_listing;


trusted_lead_count,listings_receiving_leads,average_leads_per_listing
118000,37905,3.11


### 9.6 KPI 6 — Average Days on Market

Uses the status-aware measure computed in `fact_listing`
(`days_on_market_status_aware`): completion-based age for completed
listings, current-age-from-creation for everything else.

In [0]:
%sql
SELECT ROUND(AVG(days_on_market_status_aware), 1) AS average_days_on_market
FROM gold_fact_listing
WHERE days_on_market_status_aware IS NOT NULL;


average_days_on_market
440.3


### 9.7 KPI 7 — Stale Listing Rate

**Numerator:** active listings whose age (from `listing_created_date` to
today) exceeds the declared `stale_listing_threshold_days` parameter
(Section 8.1 — currently a **90-day placeholder**, confirm with mentors).
**Denominator:** all active listings.

In [0]:
%sql
WITH active AS (
  SELECT *,
    datediff(current_date(), listing_created_date) AS current_age_days
  FROM gold_fact_listing
  WHERE listing_status = 'active'
),
stale AS (
  SELECT COUNT(*) AS stale_count FROM active
  WHERE current_age_days > 90  -- replace 90 with stale_listing_threshold_days if DECLARE VARIABLE is supported in your runtime
),
total AS (
  SELECT COUNT(*) AS active_count FROM active
)
SELECT
  s.stale_count,
  t.active_count,
  ROUND(100.0 * s.stale_count / NULLIF(t.active_count, 0), 2) AS stale_listing_rate_pct
FROM stale s CROSS JOIN total t;


stale_count,active_count,stale_listing_rate_pct
23406,23406,100.00


**Placeholder flagged again here:** the literal `90` above must match Section 8.1's declared value — if the mentor-approved threshold differs, update both places.

### 9.8 KPI 8 — DQ Pass Rate (by source file and batch, per entity)

**Numerator:** Trusted rows. **Denominator:** total Candidate rows evaluated
by Week 6 (Trusted + Quarantine, i.e. every physical record that went
through DQ). Reported per entity, `_source_file_name` and `batch_id`, per
the playbook's explicit control ("report by source file and batch").

In [0]:
%sql
WITH per_entity AS (
  SELECT 'listings' AS entity, _source_file_name, batch_id, dq_status
  FROM silver_listings_trusted
  UNION ALL
  SELECT 'listings', _source_file_name, batch_id, dq_status FROM quarantine_listings
)
SELECT entity, _source_file_name, batch_id,
       SUM(CASE WHEN dq_status = 'PASS' THEN 1 ELSE 0 END) AS trusted_rows,
       COUNT(*) AS total_evaluated_rows,
       ROUND(100.0 * SUM(CASE WHEN dq_status = 'PASS' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS dq_pass_rate_pct
FROM per_entity
GROUP BY entity, _source_file_name, batch_id
ORDER BY entity, _source_file_name, batch_id;


entity,_source_file_name,batch_id,trusted_rows,total_evaluated_rows,dq_pass_rate_pct
listings,listings.parquet,BATCH-2026-01,49000,50200,97.61


This cell covers Listings only, to keep the pattern readable. Repeat the same `UNION ALL` structure for `leads`, `localities` and `brokers` (their Trusted/Quarantine table pairs), then `UNION ALL` all four together for the full DQ Pass Rate report.

In [0]:
%sql
WITH all_entities AS (
  SELECT 'listings' AS entity, _source_file_name, batch_id, dq_status FROM silver_listings_trusted
  UNION ALL SELECT 'listings', _source_file_name, batch_id, dq_status FROM quarantine_listings
  UNION ALL SELECT 'leads', _source_file_name, batch_id, dq_status FROM silver_leads_trusted
  UNION ALL SELECT 'leads', _source_file_name, batch_id, dq_status FROM quarantine_leads
  UNION ALL SELECT 'localities', _source_file_name, batch_id, dq_status FROM silver_localities_trusted
  UNION ALL SELECT 'localities', _source_file_name, batch_id, dq_status FROM quarantine_localities
  UNION ALL SELECT 'brokers', _source_file_name, batch_id, dq_status FROM silver_brokers_trusted
  UNION ALL SELECT 'brokers', _source_file_name, batch_id, dq_status FROM quarantine_brokers
)
SELECT entity, _source_file_name, batch_id,
       SUM(CASE WHEN dq_status = 'PASS' THEN 1 ELSE 0 END) AS trusted_rows,
       COUNT(*) AS total_evaluated_rows,
       ROUND(100.0 * SUM(CASE WHEN dq_status = 'PASS' THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 2) AS dq_pass_rate_pct
FROM all_entities
GROUP BY entity, _source_file_name, batch_id
ORDER BY entity, _source_file_name, batch_id;


entity,_source_file_name,batch_id,trusted_rows,total_evaluated_rows,dq_pass_rate_pct
brokers,brokers.csv,BATCH-2026-01,320,320,100.00
leads,leads.csv,BATCH-2026-01,118000,120800,97.68
listings,listings.parquet,BATCH-2026-01,49000,50200,97.61
localities,localities.json,BATCH-2026-01,80,80,100.00


**Run and record actual result.** Do not copy a mentor-only expected value into the repository, per the playbook's explicit instruction.

## 10. Validate Gold keys and measures

In [0]:
%sql
SELECT
  SUM(CASE WHEN record_uid IS NULL THEN 1 ELSE 0 END) AS null_record_uid,
  SUM(CASE WHEN asking_price_inr < 0 THEN 1 ELSE 0 END) AS negative_price,
  SUM(CASE WHEN built_up_area_sqft < 0 THEN 1 ELSE 0 END) AS negative_area
FROM gold_fact_listing;


null_record_uid,negative_price,negative_area
0,0,0


In [0]:
%sql
SELECT record_uid, COUNT(*) AS occurrences
FROM gold_fact_listing
GROUP BY record_uid
HAVING COUNT(*) > 1;


record_uid,occurrences


**Expected result:** all zero / zero rows.

## 11. Reconcile Gold measures to eligible Trusted Silver

Aggregation reduces row count, so compare **counts/sums used by KPIs**
against the eligible Trusted detail — not Gold row count against Silver row
count.

In [0]:
%sql
WITH measure_check AS (
  SELECT
    (SELECT COUNT(*) FROM silver_listings_trusted) AS trusted_listing_rows,
    (SELECT COUNT(*) FROM gold_fact_listing) AS fact_listing_rows,
    (SELECT COUNT(*) FROM silver_leads_trusted) AS trusted_lead_rows,
    (SELECT COUNT(*) FROM gold_fact_lead) AS fact_lead_rows
)
SELECT *,
       CASE WHEN trusted_listing_rows = fact_listing_rows
             AND trusted_lead_rows = fact_lead_rows
            THEN 'PASS' ELSE 'CHECK' END AS reconciliation_status
FROM measure_check;


trusted_listing_rows,fact_listing_rows,trusted_lead_rows,fact_lead_rows,reconciliation_status
49000,49000,118000,118000,PASS


**Expected result:** `PASS`. A `CHECK` means scope, join or aggregation logic in Sections 6–7 needs investigation before the KPIs in Section 9 can be trusted.

## 12. Build the five approved summaries

**Grain declared as a design choice** (the playbook names these five
objects but does not publish their exact grain) — each is built at
`locality_id` grain except `broker_performance_summary` (`broker_id` grain),
since locality and broker are the two governed dimensions the KPI Catalogue
repeatedly reports against (median price, conversion, stale rate, days on
market). Confirm with mentors if a different grain (e.g. locality + month)
was approved, and adjust the `GROUP BY` accordingly.

### 12.1 `locality_price_summary`

In [0]:
%sql
CREATE OR REPLACE TABLE gold_locality_price_summary
USING DELTA
AS
SELECT
  fl.locality_id,
  dl.locality_name,
  dl.city,
  dl.city_zone,
  dl.market_segment,
  COUNT(DISTINCT fl.record_uid) AS total_listings,
  COUNT(DISTINCT CASE WHEN fl.listing_status = 'active' THEN fl.record_uid END) AS active_listings,
  percentile_approx(fl.asking_price_inr, 0.5) AS median_listing_price,
  percentile_approx(fl.price_per_sqft, 0.5) AS median_price_per_sqft,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM gold_fact_listing fl
LEFT JOIN gold_dim_locality dl ON fl.locality_id = dl.locality_id
GROUP BY fl.locality_id, dl.locality_name, dl.city, dl.city_zone, dl.market_segment;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*) AS summary_rows, SUM(total_listings) AS sum_total_listings FROM gold_locality_price_summary;


summary_rows,sum_total_listings
80,49000


In [0]:
%sql
SELECT
  (SELECT SUM(total_listings) FROM gold_locality_price_summary) AS summary_total,
  (SELECT COUNT(*) FROM gold_fact_listing WHERE locality_id IS NOT NULL) AS fact_total,
  CASE WHEN (SELECT SUM(total_listings) FROM gold_locality_price_summary)
          = (SELECT COUNT(*) FROM gold_fact_listing WHERE locality_id IS NOT NULL)
       THEN 'PASS' ELSE 'CHECK' END AS reconciliation_status;


summary_total,fact_total,reconciliation_status
49000,49000,PASS


### 12.2 `listing_performance_summary`

In [0]:
%sql
CREATE OR REPLACE TABLE gold_listing_performance_summary
USING DELTA
AS
SELECT
  fl.locality_id,
  COUNT(DISTINCT fl.record_uid) AS total_listings,
  ROUND(AVG(fl.days_on_market_status_aware), 1) AS avg_days_on_market,
  COUNT(DISTINCT CASE
    WHEN fl.listing_status = 'active'
         AND datediff(current_date(), fl.listing_created_date) > 90
    THEN fl.record_uid END) AS stale_listing_count,
  COUNT(DISTINCT CASE WHEN fl.listing_status = 'active' THEN fl.record_uid END) AS active_listing_count,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM gold_fact_listing fl
GROUP BY fl.locality_id;


num_affected_rows,num_inserted_rows


`stale_listing_count` uses the same 90-day placeholder threshold as KPI 7 (Section 9.7) — keep both in sync.

In [0]:
%sql
SELECT * FROM gold_listing_performance_summary ORDER BY locality_id LIMIT 20;


locality_id,total_listings,avg_days_on_market,stale_listing_count,active_listing_count,_gold_created_at,_gold_schema_version
LOC-001,565,455.9,270,270,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-002,572,432.9,276,276,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-003,582,440.8,271,271,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-004,638,433.2,308,308,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-005,589,441.2,281,281,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-006,591,462.9,311,311,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-007,606,445.9,301,301,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-008,586,433.2,273,273,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-009,645,448.4,296,296,2026-09-05T14:10:16.638Z,propiq_gold_v1.0
LOC-010,585,442.8,275,275,2026-09-05T14:10:16.638Z,propiq_gold_v1.0


### 12.3 `lead_conversion_summary`

In [0]:
%sql
CREATE OR REPLACE TABLE gold_lead_conversion_summary
USING DELTA
AS
WITH listing_lead_stats AS (
  SELECT
    fl.locality_id,
    fl.record_uid,
    fl.is_completed,
    COUNT(fd.record_uid) AS lead_count,
    SUM(CASE WHEN fd.qualified_flag = true THEN 1 ELSE 0 END) AS qualified_lead_count
  FROM gold_fact_listing fl
  LEFT JOIN gold_fact_lead fd ON fl.listing_id = fd.listing_id
  GROUP BY fl.locality_id, fl.record_uid, fl.is_completed
)
SELECT
  locality_id,
  COUNT(DISTINCT CASE WHEN lead_count > 0 THEN record_uid END) AS listings_receiving_leads,
  SUM(lead_count) AS total_leads,
  ROUND(SUM(lead_count) * 1.0 / NULLIF(COUNT(DISTINCT CASE WHEN lead_count > 0 THEN record_uid END), 0), 2)
    AS avg_leads_per_listing,
  COUNT(DISTINCT CASE WHEN qualified_lead_count > 0 THEN record_uid END) AS listings_with_qualified_leads,
  COUNT(DISTINCT CASE WHEN qualified_lead_count > 0 AND is_completed = true THEN record_uid END)
    AS closed_after_qualified_lead,
  ROUND(100.0
    * COUNT(DISTINCT CASE WHEN qualified_lead_count > 0 AND is_completed = true THEN record_uid END)
    / NULLIF(COUNT(DISTINCT CASE WHEN qualified_lead_count > 0 THEN record_uid END), 0), 2)
    AS lead_conversion_rate_pct,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM listing_lead_stats
GROUP BY locality_id;


num_affected_rows,num_inserted_rows


**Join-safety note:** this summary intentionally joins `fact_listing` to
`fact_lead` and immediately re-aggregates back to listing grain
(`GROUP BY ... record_uid`) inside the CTE before rolling up to
`locality_id` — this is exactly the "aggregate the child first" recovery
pattern the playbook names for the "counts unexpectedly increase" failure
mode, applied deliberately rather than as a bug fix.

In [0]:
%sql
SELECT * FROM gold_lead_conversion_summary ORDER BY locality_id LIMIT 20;


locality_id,listings_receiving_leads,total_leads,avg_leads_per_listing,listings_with_qualified_leads,closed_after_qualified_lead,lead_conversion_rate_pct,_gold_created_at,_gold_schema_version
LOC-001,423,1295,3.06,306,79,25.82,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-002,460,1414,3.07,343,103,30.03,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-003,457,1455,3.18,350,97,27.71,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-004,488,1539,3.15,367,127,34.60,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-005,445,1358,3.05,332,97,29.22,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-006,488,1494,3.06,352,90,25.57,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-007,465,1525,3.28,366,101,27.60,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-008,449,1394,3.10,324,108,33.33,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-009,487,1501,3.08,364,109,29.95,2026-09-05T14:10:30.465Z,propiq_gold_v1.0
LOC-010,465,1422,3.06,348,95,27.30,2026-09-05T14:10:30.465Z,propiq_gold_v1.0


### 12.4 `broker_performance_summary`

In [0]:
%sql
CREATE OR REPLACE TABLE gold_broker_performance_summary
USING DELTA
AS
SELECT
  fl.broker_id,
  db.agency_name,
  db.broker_tier,
  COUNT(DISTINCT fl.record_uid) AS total_listings,
  COUNT(DISTINCT CASE WHEN fl.listing_status = 'active' THEN fl.record_uid END) AS active_listings,
  percentile_approx(fl.asking_price_inr, 0.5) AS median_listing_price,
  ROUND(AVG(fl.days_on_market_status_aware), 1) AS avg_days_on_market,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM gold_fact_listing fl
LEFT JOIN gold_dim_broker db ON fl.broker_id = db.broker_id
GROUP BY fl.broker_id, db.agency_name, db.broker_tier;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
  (SELECT SUM(total_listings) FROM gold_broker_performance_summary) AS summary_total,
  (SELECT COUNT(*) FROM gold_fact_listing WHERE broker_id IS NOT NULL) AS fact_total,
  CASE WHEN (SELECT SUM(total_listings) FROM gold_broker_performance_summary)
          = (SELECT COUNT(*) FROM gold_fact_listing WHERE broker_id IS NOT NULL)
       THEN 'PASS' ELSE 'CHECK' END AS reconciliation_status;


summary_total,fact_total,reconciliation_status
49000,49000,PASS


### 12.5 `inventory_age_summary`

In [0]:
%sql
CREATE OR REPLACE TABLE gold_inventory_age_summary
USING DELTA
AS
SELECT
  fl.locality_id,
  COUNT(DISTINCT CASE WHEN fl.listing_status = 'active' THEN fl.record_uid END) AS active_listings,
  ROUND(AVG(CASE WHEN fl.listing_status = 'active'
                 THEN fl.days_on_market_status_aware END), 1) AS avg_active_days_on_market,
  COUNT(DISTINCT CASE
    WHEN fl.listing_status = 'active'
         AND datediff(current_date(), fl.listing_created_date) > 90
    THEN fl.record_uid END) AS stale_listing_count,
  ROUND(100.0
    * COUNT(DISTINCT CASE
        WHEN fl.listing_status = 'active'
             AND datediff(current_date(), fl.listing_created_date) > 90
        THEN fl.record_uid END)
    / NULLIF(COUNT(DISTINCT CASE WHEN fl.listing_status = 'active' THEN fl.record_uid END), 0), 2)
    AS stale_listing_rate_pct,
  current_timestamp() AS _gold_created_at,
  'propiq_gold_v1.0' AS _gold_schema_version
FROM gold_fact_listing fl
GROUP BY fl.locality_id;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_inventory_age_summary ORDER BY locality_id LIMIT 20;


locality_id,active_listings,avg_active_days_on_market,stale_listing_count,stale_listing_rate_pct,_gold_created_at,_gold_schema_version
LOC-001,270,573.5,270,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-002,276,551.5,276,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-003,271,578.1,271,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-004,308,577.2,308,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-005,281,567.7,281,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-006,311,560.8,311,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-007,301,576.2,301,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-008,273,545.0,273,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-009,296,580.8,296,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0
LOC-010,275,579.4,275,100.00,2026-09-05T14:10:58.759Z,propiq_gold_v1.0


## 13. Manual spot-checks (one locality, one broker, one listing)

The playbook's Week-7 objective explicitly requires spot-checking at least
one locality, one broker and one listing manually — not trusting the
aggregation blind.

In [0]:
%sql
SELECT * FROM gold_locality_price_summary ORDER BY total_listings DESC LIMIT 1;


locality_id,locality_name,city,city_zone,market_segment,total_listings,active_listings,median_listing_price,median_price_per_sqft,_gold_created_at,_gold_schema_version
LOC-064,Chennai North Cluster 04,Chennai,North,Luxury,670,316,25602104,17027,2026-09-05T14:09:53.541Z,propiq_gold_v1.0


In [0]:
%sql
-- Replace 'PUT_A_REAL_LOCALITY_ID_HERE' with the locality_id from the row above,
-- then manually verify the median price and listing count against fact_listing directly.
SELECT COUNT(*) AS manual_listing_count,
       percentile_approx(asking_price_inr, 0.5) AS manual_median_price
FROM gold_fact_listing
WHERE locality_id = 'PUT_A_REAL_LOCALITY_ID_HERE';


manual_listing_count,manual_median_price
0,null


In [0]:
%sql
SELECT * FROM gold_broker_performance_summary ORDER BY total_listings DESC LIMIT 1;


broker_id,agency_name,broker_tier,total_listings,active_listings,median_listing_price,avg_days_on_market,_gold_created_at,_gold_schema_version
BRK-0204,Propiq Synthetic Agency 204,Premier,193,79,13863855,413.2,2026-09-05T14:10:44.835Z,propiq_gold_v1.0


In [0]:
%sql
-- Replace 'PUT_A_REAL_LISTING_ID_HERE' with any real listing_id and manually
-- confirm every fact_listing/fact_lead field traces back to Trusted Silver.
SELECT * FROM gold_fact_listing WHERE listing_id = 'PUT_A_REAL_LISTING_ID_HERE';


record_uid,listing_id,locality_id,broker_id,property_key,price_band_key,listing_status,asking_price_inr,built_up_area_sqft,price_per_sqft,calculated_price_per_sqft,price_per_sqft_variance,listing_created_date,completion_date,last_updated_timestamp,is_completed,is_chronology_valid,actual_days_on_market,days_since_last_update,days_on_market_status_aware,_gold_created_at,_gold_schema_version


In [0]:
%sql
SELECT * FROM gold_fact_lead WHERE listing_id = 'PUT_A_REAL_LISTING_ID_HERE';


record_uid,lead_id,listing_id,lead_channel_key,buyer_intent,qualified_flag,lead_status,budget_band,lead_timestamp,_gold_created_at,_gold_schema_version


**Run and record actual result** with real IDs from your workspace — do not leave the placeholder text in committed evidence.

## 14. Controlled repeat-run proof

Preserve a business-column baseline, rerun the build, then diff both
directions — technical `_gold_created_at` timestamps may change; business
rows and measures must not.

In [0]:
%sql
CREATE OR REPLACE TABLE fact_listing_rerun_baseline
USING DELTA
AS
SELECT record_uid, listing_id, locality_id, broker_id, property_key, price_band_key,
       listing_status, asking_price_inr, built_up_area_sqft, price_per_sqft,
       is_completed, days_on_market_status_aware
FROM gold_fact_listing;


num_affected_rows,num_inserted_rows


**Action:** rerun Sections 6 and 12 (the `fact_listing` build and every summary that reads it), then compare.

In [0]:
%sql
WITH baseline_minus_current AS (
  SELECT * FROM fact_listing_rerun_baseline
  EXCEPT
  SELECT record_uid, listing_id, locality_id, broker_id, property_key, price_band_key,
         listing_status, asking_price_inr, built_up_area_sqft, price_per_sqft,
         is_completed, days_on_market_status_aware
  FROM gold_fact_listing
),
current_minus_baseline AS (
  SELECT record_uid, listing_id, locality_id, broker_id, property_key, price_band_key,
         listing_status, asking_price_inr, built_up_area_sqft, price_per_sqft,
         is_completed, days_on_market_status_aware
  FROM gold_fact_listing
  EXCEPT
  SELECT * FROM fact_listing_rerun_baseline
)
SELECT
  (SELECT COUNT(*) FROM baseline_minus_current) + (SELECT COUNT(*) FROM current_minus_baseline)
  AS changed_business_rows;


changed_business_rows
0


**Expected result:** `changed_business_rows = 0`.

In [0]:
%sql
DROP TABLE fact_listing_rerun_baseline;


**Cleanup:** remove the short-lived comparison table after capturing evidence.

## 15. Common failures and recovery

| Symptom | Likely cause | Recovery |
|---|---|---|
| Broker/locality listing counts look inflated | `fact_listing` was joined to `fact_lead` before aggregating | Rebuild `fact_listing` from Section 6 only — it must never join leads |
| Join row count increases in Section 6/7 | Duplicate lookup key in a dimension | Re-run Section 4's uniqueness checks; repair Week 6 if a duplicate slipped through |
| `unmatched_price_band_rows` nonzero | `asking_price_inr` outside the declared `dim_price_band` boundaries | Confirm the P15-DQ-03 range is enforced upstream; widen bands only with mentor approval |
| KPI shows `NULL` unexpectedly | Zero-denominator policy (Section 8.1) — this is the documented behaviour, not a bug | Confirm with mentors whether `0` was actually approved instead |
| Summary total doesn't match `fact_listing` count | A `GROUP BY` key changed grain unexpectedly, or a `LEFT JOIN` produced extra rows | Re-run the reconciliation cell directly under the summary in question |
| Rerun shows changed business rows | Non-deterministic tie-breaking in a `row_number()` dimension key, or an upstream Trusted table changed between runs | Confirm Trusted tables are unchanged; add explicit tie-breaking to `ORDER BY` in `dim_property`/`dim_listing_status`/`dim_lead_channel` |


## 16. GitHub evidence (per the playbook's Week-7 evidence table)

| Exact path | Required evidence |
|---|---|
| `notebooks/05_gold_aggregations.ipynb` | this completed notebook |
| `docs/gold_metrics_definition.md` | the 8 KPI contracts (Section 8–9), the declared price bands (5.5) and stale-listing threshold (8.1/9.7), confirmed with mentors |
| `silver_*_trusted` tables | unchanged Week-6 outputs this notebook reads |
| `P15-D05.png` | screenshot evidence per the playbook's Week-7 file list |
| `weekly_logs/week07_log.md` | outcome, blockers, validation, ownership, rework, AI Transparency Note |

**Suggested commits** (from the playbook):
`week07: implement gold model, kpis and reconciliation`;
`week07: add PropIQ validation and reconciliation evidence`;
`week07: close mentor rework and document recovery`.


## 17. Week-7 exit checklist

- [ ] All objectives are demonstrated with working artifacts, not screenshots alone.
- [ ] Exact paths and object names match the playbook (`fact_listing`, `fact_lead`, the 7 `dim_*`, the 5 named summaries).
- [ ] `fact_listing` was built **without** joining `silver_leads_trusted`.
- [ ] Every join-safety and reference-control check in Sections 4, 6.3, 6.6, 7.2 passed.
- [ ] Reconciliation in Section 11 (and each summary's own reconciliation in Section 12) is `PASS`.
- [ ] All 8 KPIs are computed with a stated contract, not invented ad hoc.
- [ ] The stale-listing threshold and zero-denominator policy are flagged as placeholders needing mentor confirmation, not silently treated as final.
- [ ] Manual spot-checks (Section 13) were run with real IDs, not left as placeholder text.
- [ ] Controlled rerun (Section 14) shows zero changed business rows.
- [ ] No Power BI, streaming Gold facts, or invented KPI rules are included — those belong to Week 8+ and Week 10.


## 18. Viva and mentor-review questions

1. Why do `fact_listing` and `fact_lead` remain separate? *(the playbook's own set viva question for this week)*
2. What would happen to broker/locality listing counts if `fact_listing` were joined to `fact_lead` before aggregating?
3. Why is `loan_id`-style per-lead detail absent from the locality/broker summaries?
4. Why does Median Price per Sq Ft get computed directly from `fact_listing`, not from a joined summary?
5. Why does Average Days on Market use a status-aware end date instead of always using `actual_days_on_market`?
6. What does `NULL` mean in a KPI's output here, and why was that chosen over `0`?
7. Where did the 90-day stale-listing threshold and the price-band boundaries come from, and why must they be confirmed with mentors before being treated as final?
8. How did you prove that the `fact_lead` → `fact_listing` join in Section 7 did not lose or duplicate any lead?
9. Why does `lead_conversion_summary` aggregate to listing grain inside a CTE before rolling up to locality grain?
10. What must change to extend this notebook to `fact_listing_status_event` in Week 10?


## 19. Final boundary

**Week 7 includes:** KPI definitions, Gold dimension/fact/summary design, a
complete build of `fact_listing`, `fact_lead`, all 7 dimensions, all 5
summaries and all 8 KPIs, validation and evidence.

**Week 7 does not include:** Power BI dashboards (Week 8+), streaming Gold
facts (Week 10), or KPI answers/thresholds copied from a mentor-only
reference instead of your own executed results.
